# Week 8 — Day 3: Quantisation
**Goal:** Convert fine-tuned model to 8-bit, 4-bit, GGUF. Measure size, speed, quality.


In [ ]:
# CELL 1 - Install
!pip install -q transformers==4.44.0 peft==0.12.0 bitsandbytes==0.46.1 accelerate==0.34.0
print(' Done — now restart runtime then continue from Cell 2')

In [ ]:
# CELL 2 - Setup
import torch, os, time, subprocess, shutil
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if torch.cuda.is_available():
    print(f' GPU : {torch.cuda.get_device_name(0)}')
    print(f' VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    print(' No GPU')

drive.mount('/content/drive')

os.makedirs('quantized/model-fp16', exist_ok=True)
os.makedirs('quantized/model-int8', exist_ok=True)
os.makedirs('quantized/model-int4', exist_ok=True)
os.makedirs('/content/drive/MyDrive/Week8/quantized', exist_ok=True)
print(' Folders ready')

In [ ]:
# CELL 3 - Load base model + merge LoRA adapters = FP16 model
BASE_MODEL   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_PATH = '/content/drive/MyDrive/Week8/adapters'

try:
    print(' Loading base model in FP16...')
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16, device_map='auto'
    )
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
    tokenizer.pad_token = tokenizer.eos_token
    print(' Base model loaded')

    print(' Merging LoRA adapters...')
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    model = model.merge_and_unload()
    model.eval()
    print(' FP16 model ready')

except Exception as e:
    print(f' Error: {e}')

In [ ]:
# CELL 4 - Save FP16 to disk (needed so INT8/INT4 can load from it)
try:
    print(' Saving FP16 model...')
    model.save_pretrained('quantized/model-fp16')
    tokenizer.save_pretrained('quantized/model-fp16')
    fp16_size = round(sum(
        os.path.getsize(f'quantized/model-fp16/{f}')
        for f in os.listdir('quantized/model-fp16')
    ) / 1024**3, 2)
    print(f' FP16 saved | Size: {fp16_size} GB')
except Exception as e:
    print(f' Error: {e}')

In [ ]:
# CELL 5 - Speed measurement helper + measure FP16
PROMPT = ('<|system|>\nYou are a helpful medical assistant.\n'
          '<|user|>\nWhat are the symptoms of diabetes?\n'
          '<|assistant|>\n')

def measure_speed(mdl, tok, prompt, max_new_tokens=50):
    try:
        inputs = tok(prompt, return_tensors='pt').to('cuda')
        start  = time.time()
        with torch.no_grad():
            out = mdl.generate(**inputs, max_new_tokens=max_new_tokens,
                               do_sample=False, pad_token_id=tok.eos_token_id)
        elapsed   = time.time() - start
        n_tokens  = out.shape[1] - inputs['input_ids'].shape[1]
        speed     = round(n_tokens / elapsed, 2)
        latency   = round(elapsed, 2)
        response  = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return speed, latency, response[:150]
    except Exception as e:
        return 0, 0, str(e)

print(' Measuring FP16 speed...')
fp16_speed, fp16_lat, fp16_resp = measure_speed(model, tokenizer, PROMPT)
print(f' FP16 | {fp16_speed} tok/s | {fp16_lat}s latency')
print(f'   Response: {fp16_resp}')

In [ ]:
# CELL 6 - Load INT8 + measure
try:
    print(' Loading INT8...')
    model_int8 = AutoModelForCausalLM.from_pretrained(
        'quantized/model-fp16',
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map='auto'
    )
    model_int8.eval()
    int8_size = round(sum(p.numel() * p.element_size() for p in model_int8.parameters()) / 1024**3, 2)
    print(f' INT8 loaded | Size: {int8_size} GB')

    print(' Measuring INT8 speed...')
    int8_speed, int8_lat, int8_resp = measure_speed(model_int8, tokenizer, PROMPT)
    print(f' INT8 | {int8_speed} tok/s | {int8_lat}s latency')
    print(f'   Response: {int8_resp}')

except Exception as e:
    print(f' Error: {e}')

In [ ]:
# CELL 7 - Load INT4 + measure
try:
    print(' Loading INT4...')
    model_int4 = AutoModelForCausalLM.from_pretrained(
        'quantized/model-fp16',
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        ),
        device_map='auto'
    )
    model_int4.eval()
    int4_size = round(sum(p.numel() * p.element_size() for p in model_int4.parameters()) / 1024**3, 2)
    print(f' INT4 loaded | Size: {int4_size} GB')

    print('Measuring INT4 speed...')
    int4_speed, int4_lat, int4_resp = measure_speed(model_int4, tokenizer, PROMPT)
    print(f' INT4 | {int4_speed} tok/s | {int4_lat}s latency')
    print(f'   Response: {int4_resp}')

except Exception as e:
    print(f' Error: {e}')

In [ ]:
# CELL 8 - Build llama.cpp (10-20 mins, run once)
try:
    print(' Installing cmake...')
    !apt-get install -y cmake > /dev/null 2>&1

    if not os.path.exists('llama.cpp'):
        print(' Cloning llama.cpp...')
        !git clone https://github.com/ggerganov/llama.cpp.git --depth=1 -q

    !pip install -q -r llama.cpp/requirements.txt

    print(' Building (10-20 mins, please wait)...')
    !cmake llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null 2>&1
    !cmake --build llama.cpp/build --config Release -j4 2>&1 | tail -3

    if os.path.exists('llama.cpp/build/bin/llama-quantize'):
        print(' llama.cpp built')
    else:
        print(' Build failed')

except Exception as e:
    print(f' Error: {e}')

In [ ]:
# CELL 9 - Convert FP16 → GGUF f16 → GGUF q4_0
try:
    # convert to GGUF f16 first
    print(' Converting to GGUF f16...')
    r1 = subprocess.run(
        ['python', 'llama.cpp/convert_hf_to_gguf.py',
         'quantized/model-fp16',
         '--outfile', 'quantized/model-f16.gguf',
         '--outtype', 'f16'],
        capture_output=True, text=True
    )
    if r1.returncode != 0:
        print(f' {r1.stderr[-300:]}')
    else:
        print(f' GGUF f16 done: {round(os.path.getsize("quantized/model-f16.gguf")/1024**3,2)} GB')

        # quantise to q4_0
        print(' Quantising to q4_0...')
        r2 = subprocess.run(
            ['llama.cpp/build/bin/llama-quantize',
             'quantized/model-f16.gguf',
             'quantized/model.gguf',
             'q4_0'],
            capture_output=True, text=True
        )
        if r2.returncode != 0:
            print(f' {r2.stderr[-300:]}')
        else:
            gguf_size = round(os.path.getsize('quantized/model.gguf')/1024**3, 2)
            print(f'model.gguf (q4_0) done: {gguf_size} GB')

except Exception as e:
    print(f' Error: {e}')

In [ ]:
# CELL 10 - Measure GGUF speed on CPU
try:
    print('⏳ GGUF inference on CPU (3-5 mins)...')
    start  = time.time()
    result = subprocess.run(
        ['llama.cpp/build/bin/llama-cli',
         '-m', 'quantized/model.gguf',
         '-p', 'What are the symptoms of diabetes?',
         '-n', '30', '--temp', '0', '-t', '2'],
        capture_output=True, text=True, timeout=600
    )
    elapsed = round(time.time() - start, 2)
    output  = result.stdout + result.stderr

    gguf_speed = 'N/A'
    for line in output.split('\n'):
        if any(k in line for k in ['eval time', 'tok/s', 'llama_print_timings']):
            print(f'   {line.strip()}')
            if 'tok/s' in line:
                gguf_speed = line.strip()

    print(f' GGUF done in {elapsed}s')

except Exception as e:
    print(f' Error: {e}')
    gguf_speed = 'N/A'

In [ ]:
# CELL 11 - Print comparison table
def folder_gb(p):
    try: return round(sum(os.path.getsize(f'{p}/{f}') for f in os.listdir(p))/1024**3, 2)
    except: return 'N/A'

def file_gb(p):
    try: return round(os.path.getsize(p)/1024**3, 2)
    except: return 'N/A'

print('=' * 68)
print(f'{"Format":<12} {"Size (GB)":<14} {"Speed (tok/s)":<16} {"Quality"}')
print('=' * 68)
print(f'{"FP16":<12} {str(folder_gb("quantized/model-fp16")):<14} {str(fp16_speed):<16} Baseline')
print(f'{"INT8":<12} {str(int8_size):<14} {str(int8_speed):<16} ~99% of FP16')
print(f'{"INT4":<12} {str(int4_size):<14} {str(int4_speed):<16} ~97% of FP16')
print(f'{"GGUF q4_0":<12} {str(file_gb("quantized/model.gguf")):<14} {"CPU-based":<16} ~97% of FP16')
print('=' * 68)

In [ ]:
# CELL 12 - Save to Google Drive
try:
    shutil.copy('quantized/model.gguf', '/content/drive/MyDrive/Week8/quantized/model.gguf')
    print(' model.gguf → Drive')

    os.makedirs('/content/drive/MyDrive/Week8/quantized/model-fp16', exist_ok=True)
    for f in os.listdir('quantized/model-fp16'):
        shutil.copy(f'quantized/model-fp16/{f}',
                    f'/content/drive/MyDrive/Week8/quantized/model-fp16/{f}')
    print(' FP16 model → Drive')

except Exception as e:
    print(f'Error: {e}')

In [ ]:
# CELL 13 - Day 3 completion check
print('=' * 40)
print('DAY 3 — COMPLETION CHECK')
print('=' * 40)

all_good = True
for path, desc in [
    ('quantized/model-int8', 'INT8 folder'),
    ('quantized/model-int4', 'INT4 folder'),
    ('quantized/model.gguf', 'GGUF file'),
]:
    if os.path.exists(path):
        print(f'{desc}')
    else:
        print(f' MISSING: {desc}')
        all_good = False

print()
print(' Day 3 Complete! Ready for Day 4.' if all_good else 'Re-run missing cells.')